# 03 — Mel Spectrogram Extraction

Visualize what MERT 'sees' before we feed it audio.

This notebook is educational — it shows the mel spectrogram pipeline
and lets you compare spectrograms across songs to build intuition
for why perceptual similarity should work.

**Requires**: fma_small audio files in `../data/audio/fma_small/`
Or just use personal MP3s from `../data/audio/personal/`.

In [ ]:
import sys
sys.path.insert(0, '..')

import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from anther_ml.features import extract_mel_spectrogram, SR, N_MELS, HOP_LENGTH
from anther_ml.data import get_audio_path

AUDIO_DIR = '../data/audio/fma_small'
PERSONAL_DIR = '../data/audio/personal'

print('imports ok')

## Visualize a single spectrogram

In [ ]:
# Use a personal MP3 or any FMA track
# Uncomment one:
# song_path = get_audio_path(AUDIO_DIR, track_id=2)  # FMA track 000002
song_path = list(Path(PERSONAL_DIR).glob('*.mp3'))[0]  # first personal MP3

print(f'Loading: {song_path}')
S = extract_mel_spectrogram(song_path)
print(f'Spectrogram shape: {S.shape}  (mels x time_frames)')

fig, ax = plt.subplots(figsize=(14, 4))
img = librosa.display.specshow(
    S, sr=SR, hop_length=HOP_LENGTH,
    x_axis='time', y_axis='mel', ax=ax
)
fig.colorbar(img, ax=ax, format='%+2.0f dB')
ax.set_title(f'Log-mel spectrogram: {Path(song_path).stem}')
plt.tight_layout()
plt.show()

## Compare two songs side-by-side

In [ ]:
personal_songs = list(Path(PERSONAL_DIR).glob('*.mp3'))

if len(personal_songs) < 2:
    print('Need at least 2 personal MP3s. Add more to data/audio/personal/')
else:
    paths = personal_songs[:2]
    fig, axes = plt.subplots(1, 2, figsize=(18, 4))
    for ax, path in zip(axes, paths):
        S = extract_mel_spectrogram(path)
        img = librosa.display.specshow(
            S, sr=SR, hop_length=HOP_LENGTH,
            x_axis='time', y_axis='mel', ax=ax
        )
        fig.colorbar(img, ax=ax, format='%+2.0f dB')
        ax.set_title(Path(path).stem)
    plt.suptitle('Side-by-side spectrograms — do these look similar to you?')
    plt.tight_layout()
    plt.show()

## What MERT actually receives

MERT's `Wav2Vec2FeatureExtractor` preprocesses the waveform internally —
it normalises and windows the raw 24kHz signal. Here's what that looks like.

In [ ]:
MERT_SR = 24000

y, _ = librosa.load(str(personal_songs[0]), sr=MERT_SR, mono=True, duration=30.0)
print(f'Waveform at 24kHz: {y.shape[0]} samples ({y.shape[0]/MERT_SR:.1f}s)')
print(f'Amplitude range: [{y.min():.3f}, {y.max():.3f}]')

fig, axes = plt.subplots(2, 1, figsize=(14, 5))

# Waveform
t = np.linspace(0, len(y) / MERT_SR, len(y))
axes[0].plot(t, y, linewidth=0.3, color='steelblue')
axes[0].set_title('Waveform (24kHz, 30s)')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')

# Corresponding mel spectrogram at 24kHz
S24 = librosa.feature.melspectrogram(y=y, sr=MERT_SR, n_mels=128, n_fft=1024, hop_length=256)
S24_db = librosa.power_to_db(S24, ref=np.max)
img = librosa.display.specshow(S24_db, sr=MERT_SR, hop_length=256,
                                x_axis='time', y_axis='mel', ax=axes[1])
fig.colorbar(img, ax=axes[1], format='%+2.0f dB')
axes[1].set_title('Mel spectrogram (24kHz) — for reference only, MERT uses raw waveform')

plt.tight_layout()
plt.show()

## Batch check: can we load all personal MP3s?

In [ ]:
from tqdm import tqdm

errors = []
for path in tqdm(personal_songs, desc='Checking files'):
    try:
        y, sr = librosa.load(str(path), sr=MERT_SR, mono=True, duration=5.0)
        if len(y) < MERT_SR:  # less than 1 second
            errors.append((path, 'too short'))
    except Exception as e:
        errors.append((path, str(e)))

if errors:
    print(f'\n{len(errors)} files with issues:')
    for p, msg in errors:
        print(f'  {p.name}: {msg}')
else:
    print(f'\nAll {len(personal_songs)} personal MP3s loaded cleanly.')